# Mars Forecast Analyst Case - Data Exploration

Objective: understand the weekly forecast, actual delivered cases, promotion flags,
and casefill signals before proposing a January 2022 forecast.

In [ ]:
"""Set up imports, paths, and the shared Mars forecast pipeline."""

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mars_forecast_case.pipeline import build_outputs

sns.set_theme(style="whitegrid")


In [ ]:
def load_analysis_tables(project_root: Path) -> dict[str, pd.DataFrame]:
    """Rebuild pipeline outputs and load the analyst-facing tables."""

    build_outputs(project_root / "outputs")
    outputs = project_root / "outputs"
    return {
        "weekly": pd.read_csv(outputs / "weekly_clean_data.csv", parse_dates=["week_start"]),
        "period": pd.read_csv(outputs / "period_summary.csv"),
        "promotion": pd.read_csv(outputs / "promotion_summary.csv"),
        "forecast": pd.read_csv(outputs / "forecast_summary.csv"),
    }


tables = load_analysis_tables(PROJECT_ROOT)
weekly = tables["weekly"]
period = tables["period"]
promotion = tables["promotion"]
forecast = tables["forecast"]

weekly.head()


## Initial Read

The workbook contains 52 weekly observations for one product in 2021.
The important fields are the existing forecast, delivered actuals, customer promotion flags,
and casefill. Because casefill is delivered cases divided by ordered cases, delivered actuals
should be corrected to estimated ordered demand before judging demand strength.

In [ ]:
def summarize_data_quality(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize service, promotion, and forecast accuracy indicators."""

    return pd.DataFrame(
        {
            "metric": [
                "weeks",
                "promotion weeks",
                "weeks below 95% casefill",
                "total actual delivered ('000)",
                "total estimated ordered ('000)",
                "total forecast ('000)",
                "mean absolute percentage error",
            ],
            "value": [
                len(df),
                int(df["promo_flag"].sum()),
                int((df["casefill"] < 0.95).sum()),
                round(df["actual_k_cases"].sum(), 1),
                round(df["estimated_ordered_k_cases"].sum(), 1),
                round(df["forecast_k_cases"].sum(), 1),
                f"{df['absolute_percentage_error'].mean() * 100:.1f}%",
            ],
        }
    )


summarize_data_quality(weekly)


In [ ]:
"""Visualize forecast versus delivered actuals and identify promotion weeks."""

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(weekly["week"], weekly["forecast_k_cases"], label="Forecast", color="#6B7280", linewidth=2)
ax.plot(weekly["week"], weekly["actual_k_cases"], label="Actual delivered", color="#9E1B32", linewidth=2.2)
promo = weekly[weekly["promo_flag"]]
ax.scatter(promo["week"], promo["actual_k_cases"], label="Promotion week", color="#F2A900", edgecolor="#263238", s=55, zorder=5)
ax.set_title("Weekly forecast versus actual delivered cases")
ax.set_xlabel("Week")
ax.set_ylabel("'000 cases")
ax.legend(frameon=False, ncols=3, loc="upper left")
plt.show()


In [ ]:
"""Show where actuals understate ordered demand because service levels fell."""

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(weekly["week"], weekly["casefill_pct"], color="#007A78", linewidth=2.3)
ax.axhline(95, color="#9E1B32", linestyle="--", linewidth=1.4, label="95% service watchline")
low = weekly[weekly["casefill_pct"] < 95]
ax.scatter(low["week"], low["casefill_pct"], color="#9E1B32", s=60, label="Low casefill", zorder=5)
ax.set_ylim(65, 101)
ax.set_title("Casefill fell sharply in late 2021")
ax.set_xlabel("Week")
ax.set_ylabel("Casefill (%)")
ax.legend(frameon=False, loc="lower left")
plt.show()


In [ ]:
"""Review period-level bias, service loss, and promotion concentration."""

display(period)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(period["period"], period["estimated_ordered_k_cases"], color="#007A78", alpha=0.85, label="Estimated ordered")
ax.plot(period["period"], period["forecast_k_cases"], color="#9E1B32", marker="o", label="Forecast")
ax.set_title("Period demand after correcting delivered actuals for casefill")
ax.set_ylabel("'000 cases")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
plt.show()


In [ ]:
"""Compare promotion demand ranges by customer."""

display(promotion)

data = promotion[promotion["promotion_type"].isin(["Customer 1", "Customer 2"])].copy()
fig, ax = plt.subplots(figsize=(9, 4))
for idx, row in data.reset_index(drop=True).iterrows():
    ax.hlines(idx, row["min_ordered_k_cases"], row["max_ordered_k_cases"], color="#CBD5E1", linewidth=8)
    ax.scatter(row["median_ordered_k_cases"], idx, color="#9E1B32", s=80, zorder=4)
    ax.text(row["median_ordered_k_cases"] + 10, idx, f"median {row['median_ordered_k_cases']:.0f}", va="center")
ax.set_yticks(range(len(data)), data["promotion_type"])
ax.set_xlabel("Estimated ordered demand per promo week ('000 cases)")
ax.set_title("Customer promotion history is valuable, but variable")
plt.show()


## Exploration Takeaways

- P13 is the cleanest recent base read: no promotions and normal casefill.
- P12 delivered actuals are not a good base because service was constrained.
- Customer 1 promotions are steadier; Customer 2 has a much wider range and two extreme late-year weeks.
- January sign-off should therefore separate a factory baseline from a signed-promotion scenario.